**BIOM** (Biological Observation Matrix) is a standardized file format designed for representing biological sample by observation contingency tables, most commonly **OTU (Operational Taxonomic Unit) tables** used in microbiome research.


#### *Versions*

BIOM 1.0 (JSON) 

BIOM 2.1 (HDF5)

In [ ]:
# %pip install biom-format numpy scipy

In [3]:
import biom
import numpy as np
import pandas as pd
from biom.table import Table

#### **BIOM Table Structure**


```
              Sample1  Sample2  Sample3  Sample4
OTU_1            5        0        2        0
OTU_2            0        3        0        1
OTU_3           10        7        0        4
OTU_4            0        0        8        0
OTU_5            2        1        3        6
```

- **Observations (rows)**: Things being counted: OTUs, ASVs, taxa, genes, metabolites
- **Samples (columns)**: Biological samples: gut sample, soil sample, water sample
- **Values**: Count of each observation in each sample



| Term | Meaning |
|---|---|
| **OTU** | Operational Taxonomic Unit: a cluster of similar sequences |
| **ASV** | Amplicon Sequence Variant: exact sequence variant (higher resolution than OTU) |
| **Sparse matrix** | A matrix where most values are zero: stored efficiently |
| **Observation metadata** | Taxonomic classification, gene function, etc. |
| **Sample metadata** | Collection site, date, treatment group, patient ID, etc. |

Using the **rich sparse OTU table** [biocore/biom-format](https://github.com/biocore/biom-format) repository, has example OTU table with taxonomy and sample metadata.

In [ ]:
import urllib.request

url = 'https://raw.githubusercontent.com/biocore/biom-format/master/examples/rich_sparse_otu_table_hdf5.biom'
urllib.request.urlretrieve(url, 'rich_sparse_otu_table.biom')

table = biom.load_table('rich_sparse_otu_table.biom')
table.type

'otu table'

In [5]:
table.shape

(5, 6)

#### **IDs, Metadata, and the Data Matrix**

In [13]:
print('OTU IDs:', table.ids(axis='observation').tolist())
print('Sample IDs:', table.ids(axis='sample').tolist())

OTU IDs: ['GG_OTU_1', 'GG_OTU_2', 'GG_OTU_3', 'GG_OTU_4', 'GG_OTU_5']
Sample IDs: ['Sample1', 'Sample2', 'Sample3', 'Sample4', 'Sample5', 'Sample6']


In [14]:
for otu_id in table.ids(axis='observation'):
    meta = table.metadata(otu_id, axis='observation')
    print(f'  {otu_id}: {dict(meta)}')

  GG_OTU_1: {'taxonomy': ['k__Bacteria', 'p__Proteobacteria', 'c__Gammaproteobacteria', 'o__Enterobacteriales', 'f__Enterobacteriaceae', 'g__Escherichia', 's__']}
  GG_OTU_2: {'taxonomy': ['k__Bacteria', 'p__Cyanobacteria', 'c__Nostocophycideae', 'o__Nostocales', 'f__Nostocaceae', 'g__Dolichospermum', 's__']}
  GG_OTU_3: {'taxonomy': ['k__Archaea', 'p__Euryarchaeota', 'c__Methanomicrobia', 'o__Methanosarcinales', 'f__Methanosarcinaceae', 'g__Methanosarcina', 's__']}
  GG_OTU_4: {'taxonomy': ['k__Bacteria', 'p__Firmicutes', 'c__Clostridia', 'o__Halanaerobiales', 'f__Halanaerobiaceae', 'g__Halanaerobium', 's__Halanaerobiumsaccharolyticum']}
  GG_OTU_5: {'taxonomy': ['k__Bacteria', 'p__Proteobacteria', 'c__Gammaproteobacteria', 'o__Enterobacteriales', 'f__Enterobacteriaceae', 'g__Escherichia', 's__']}


In [15]:
for sid in table.ids(axis='sample'):
    meta = table.metadata(sid, axis='sample')
    print(f'  {sid}: {dict(meta)}')

  Sample1: {'BODY_SITE': 'gut', 'BarcodeSequence': 'CGCTTATCGAGA', 'Description': 'human gut', 'LinkerPrimerSequence': 'CATGCTGCCTCCCGTAGGAGT'}
  Sample2: {'BODY_SITE': 'gut', 'BarcodeSequence': 'CATACCAGTAGC', 'Description': 'human gut', 'LinkerPrimerSequence': 'CATGCTGCCTCCCGTAGGAGT'}
  Sample3: {'BODY_SITE': 'gut', 'BarcodeSequence': 'CTCTCTACCTGT', 'Description': 'human gut', 'LinkerPrimerSequence': 'CATGCTGCCTCCCGTAGGAGT'}
  Sample4: {'BODY_SITE': 'skin', 'BarcodeSequence': 'CTCTCGGCCTGT', 'Description': 'human skin', 'LinkerPrimerSequence': 'CATGCTGCCTCCCGTAGGAGT'}
  Sample5: {'BODY_SITE': 'skin', 'BarcodeSequence': 'CTCTCTACCAAT', 'Description': 'human skin', 'LinkerPrimerSequence': 'CATGCTGCCTCCCGTAGGAGT'}
  Sample6: {'BODY_SITE': 'skin', 'BarcodeSequence': 'CTAACTACCAAT', 'Description': 'human skin', 'LinkerPrimerSequence': 'CATGCTGCCTCCCGTAGGAGT'}


#### **Viewing as DataFrame**

In [16]:
df = table.to_dataframe(dense=True)
df

,Sample1,Sample2,Sample3,Sample4,Sample5,Sample6
GG_OTU_1,0.0,0.0,1.0,0.0,0.0,0.0
GG_OTU_2,5.0,1.0,0.0,2.0,3.0,1.0
GG_OTU_3,0.0,0.0,1.0,4.0,0.0,2.0
GG_OTU_4,2.0,1.0,1.0,0.0,0.0,1.0
GG_OTU_5,0.0,1.0,1.0,0.0,0.0,0.0


#### **Querying the Dataframe**

In [18]:
# counts for a single sample
s1 = table.data('Sample1', axis='sample', dense=True)
print('Sample1 counts:', dict(zip(table.ids('observation'), s1.astype(int))))

Sample1 counts: {np.str_('GG_OTU_1'): np.int64(0), np.str_('GG_OTU_2'): np.int64(5), np.str_('GG_OTU_3'): np.int64(0), np.str_('GG_OTU_4'): np.int64(2), np.str_('GG_OTU_5'): np.int64(0)}


In [19]:
# one OTU across all samples
otu1 = table.data('GG_OTU_1', axis='observation', dense=True)
print('GG_OTU_1 across samples:', dict(zip(table.ids('sample'), otu1.astype(int))))

GG_OTU_1 across samples: {np.str_('Sample1'): np.int64(0), np.str_('Sample2'): np.int64(0), np.str_('Sample3'): np.int64(1), np.str_('Sample4'): np.int64(0), np.str_('Sample5'): np.int64(0), np.str_('Sample6'): np.int64(0)}


#### **Saving as JSON (1.0) and HDF5 (2.1)**

In [ ]:
import os, json
from biom.util import biom_open

# save as JSON
with open('output.json.biom', 'w') as f:
    f.write(table.to_json('biom-f'))

# save as HDF5
with biom_open('output.hdf5.biom', 'w') as f:
    table.to_hdf5(f, 'biom-f')

#### **API Reference**

| What | How |
|---|---|
| Load a file | `biom.load_table('file.biom')` |
| Save as JSON | `table.to_json('generator_name')` |
| Save as HDF5 | `table.to_hdf5(file_handle, 'generator_name')` |
| Convert to DataFrame | `table.to_dataframe(dense=True)` |
| Get sample data | `table.data('Sample1', axis='sample', dense=True)` |
| Get OTU data | `table.data('OTU_1', axis='observation', dense=True)` |
| Get metadata | `table.metadata('id', axis='sample')` |
| List IDs | `table.ids(axis='sample')` or `table.ids(axis='observation')` |
| Filter | `table.filter(fn, axis='observation', inplace=False)` |
| Normalize | `table.norm(axis='sample', inplace=False)` |
| Merge tables | `table.merge(other_table)` |
| Iterate | `for vals, id, meta in table.iter(axis='sample', dense=True):` |
| Shape | `table.shape` → `(n_observations, n_samples)` |
| Total count | `table.sum()` |
| Non-zero count | `table.nnz` |
| Density | `table.get_table_density()` |